In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [3]:
!pip install -q transformers==4.44.0
!pip install -q datasets
!pip install -q peft==0.12.0
!pip install -q trl==0.10.1
!pip install -q bitsandbytes==0.43.3
!pip install -q accelerate==0.33.0
!pip install -q sacrebleu evaluate bert-score unbabel-comet
!pip install -q sentencepiece
!pip install -q protobuf==4.25.3

In [4]:
import os
import gc
import torch
import numpy as np
import pandas as pd
import warnings

warnings.filterwarnings("ignore")

from datasets import load_dataset

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)

from peft import (
    LoraConfig,
    get_peft_model,
    TaskType,
    PeftModel,
    prepare_model_for_kbit_training,
)

from trl import SFTTrainer, SFTConfig

import evaluate
from bert_score import score as bert_score_fn

# Clear memory
gc.collect()
torch.cuda.empty_cache()

# Device
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"✅ Device: {device}")

if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✅ VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Seed
torch.manual_seed(42)
np.random.seed(42)

2026-05-15 14:41:13.580355: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778856074.006427     217 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778856074.127255     217 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778856075.103212     217 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778856075.103284     217 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778856075.103287     217 computation_placer.cc:177] computation placer alr

✅ Device: cuda
✅ GPU: Tesla T4
✅ VRAM: 15.6 GB


In [5]:
print("⏳ Đang tải dataset...")

ds = load_dataset("thainq107/iwslt2015-en-vi")

print(ds)

NUM_TRAIN = 10000
NUM_EVAL  = 500

train_raw = ds["train"].shuffle(seed=42).select(range(NUM_TRAIN))
eval_raw  = ds["validation"].select(range(min(NUM_EVAL, len(ds["validation"]))))

test_raw  = ds["test"]

print(f"✅ Train: {len(train_raw)}")
print(f"✅ Eval : {len(eval_raw)}")
print(f"✅ Test : {len(test_raw)}")

print(f"\n📌 Sample:")
print(train_raw[0])

# Auto detect columns
sample = train_raw[0]

if "translation" in sample:
    nest_key = "translation"
    langs = list(sample["translation"].keys())

    src_col = langs[0]
    tgt_col = langs[1]

else:
    nest_key = None

    keys = list(sample.keys())

    src_col = keys[0]
    tgt_col = keys[1]

print(f"\n✅ nest_key = {nest_key}")
print(f"✅ src_col  = {src_col}")
print(f"✅ tgt_col  = {tgt_col}")

⏳ Đang tải dataset...


README.md:   0%|          | 0.00/522 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/17.8M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/181k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/133317 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1268 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1268 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['en', 'vi'],
        num_rows: 133317
    })
    validation: Dataset({
        features: ['en', 'vi'],
        num_rows: 1268
    })
    test: Dataset({
        features: ['en', 'vi'],
        num_rows: 1268
    })
})
✅ Train: 10000
✅ Eval : 500
✅ Test : 1268

📌 Sample:
{'en': 'This is the blood vessels .', 'vi': 'Đây là các mạch máu .'}

✅ nest_key = None
✅ src_col  = en
✅ tgt_col  = vi


In [8]:
!pip uninstall -y bitsandbytes triton
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

print("⏳ Load tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    padding_side="left",
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("✅ Tokenizer loaded!")

print(f"⏳ Loading model: {MODEL_NAME}")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,

    device_map="auto",

    torch_dtype=torch.float16,

    trust_remote_code=True,
)

print("✅ Base model loaded!")

# Important
model.config.use_cache = False

# Gradient checkpointing
model.gradient_checkpointing_enable()

# Parameter count
total_params = sum(
    p.numel()
    for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print("\n" + "="*60)

print(f"🧠 Model: {MODEL_NAME}")

print(f"📦 Total Params     : {total_params/1e6:.2f} M")

print(f"🔥 Trainable Params : {trainable_params/1e6:.2f} M")

print("="*60)

print("\n✅ Model ready!")

Found existing installation: bitsandbytes 0.42.0
Uninstalling bitsandbytes-0.42.0:
  Successfully uninstalled bitsandbytes-0.42.0
⏳ Load tokenizer...
✅ Tokenizer loaded!
⏳ Loading model: Qwen/Qwen2.5-1.5B-Instruct


generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

✅ Base model loaded!

🧠 Model: Qwen/Qwen2.5-1.5B-Instruct
📦 Total Params     : 1543.71 M
🔥 Trainable Params : 1543.71 M

✅ Model ready!


In [9]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,

    r=8,

    lora_alpha=16,

    lora_dropout=0.05,

    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],

    bias="none",
)

model = get_peft_model(
    model,
    lora_config
)

model.print_trainable_parameters()

trainable params: 9,232,384 || all params: 1,552,946,688 || trainable%: 0.5945


In [10]:
def get_pair(example):

    if nest_key:
        return (
            example[nest_key][src_col],
            example[nest_key][tgt_col]
        )

    return example[src_col], example[tgt_col]


def format_example(example):

    src, tgt = get_pair(example)

    text = (
        f"<|im_start|>system\n"
        f"You are a professional translator. "
        f"Translate English to Vietnamese accurately.<|im_end|>\n"

        f"<|im_start|>user\n"
        f"Translate to Vietnamese:\n"
        f"{src}<|im_end|>\n"

        f"<|im_start|>assistant\n"
        f"{tgt}<|im_end|>"
    )

    return {"text": text}


print("⏳ Formatting dataset...")

train_formatted = train_raw.map(
    format_example,
    remove_columns=train_raw.column_names
)

eval_formatted = eval_raw.map(
    format_example,
    remove_columns=eval_raw.column_names
)

print("\n✅ Example formatted:")
print(train_formatted[0]["text"][:500])

⏳ Formatting dataset...


Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]


✅ Example formatted:
<|im_start|>system
You are a professional translator. Translate English to Vietnamese accurately.<|im_end|>
<|im_start|>user
Translate to Vietnamese:
This is the blood vessels .<|im_end|>
<|im_start|>assistant
Đây là các mạch máu .<|im_end|>


In [11]:
OUTPUT_DIR = "/kaggle/working/llm_mt_en_vi"

os.makedirs(OUTPUT_DIR, exist_ok=True)

sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,

    gradient_accumulation_steps=32,

    gradient_checkpointing=True,

    learning_rate=2e-4,

    weight_decay=0.01,

    warmup_ratio=0.05,

    lr_scheduler_type="cosine",

    optim="adamw_torch",

    max_seq_length=256,

    dataset_text_field="text",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    fp16=True,
    logging_dir=f"{OUTPUT_DIR}/logs",
    logging_steps=50,
    report_to="none",
    save_total_limit=2,
    seed=42,
    dataloader_num_workers=2,
)

trainer = SFTTrainer(
    model=model,

    args=sft_config,

    train_dataset=train_formatted,

    eval_dataset=eval_formatted,

    tokenizer=tokenizer,
)

print("🚀 Training...")

train_result = trainer.train()

print(f"\n✅ Training done!")

print(f"Loss: {train_result.training_loss:.4f}")

trainer.model.save_pretrained(
    f"{OUTPUT_DIR}/best_adapter"
)

tokenizer.save_pretrained(
    f"{OUTPUT_DIR}/best_adapter"
)

merged_model = trainer.model.merge_and_unload()

merged_model.save_pretrained(
    f"{OUTPUT_DIR}/merged_model"
)

tokenizer.save_pretrained(
    f"{OUTPUT_DIR}/merged_model"
)

print(f"\n✅ Model saved!")

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

🚀 Training...


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Epoch,Training Loss,Validation Loss
0,1.271900,1.221419
2,1.116900,1.229334


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 


✅ Training done!
Loss: 1.2613

✅ Model saved!


In [13]:
print("⏳ Load inference model...")

base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)

inf_model = PeftModel.from_pretrained(
    base,
    f"{OUTPUT_DIR}/best_adapter"
)

inf_model.eval()


def translate_batch(texts, batch_size=8):

    results = []

    for i in range(0, len(texts), batch_size):

        batch = texts[i:i+batch_size]

        prompts = [
            (
                f"<|im_start|>system\n"
                f"You are a professional translator. "
                f"Translate English to Vietnamese accurately.<|im_end|>\n"

                f"<|im_start|>user\n"
                f"Translate to Vietnamese:\n"
                f"{t}<|im_end|>\n"

                f"<|im_start|>assistant\n"
            )
            for t in batch
        ]

        inputs = tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=256,
        ).to(device)

        with torch.no_grad():

            outputs = inf_model.generate(
                **inputs,

                max_new_tokens=128,

                num_beams=4,

                do_sample=False,

                temperature=0.0,

                early_stopping=True,

                pad_token_id=tokenizer.eos_token_id,

                eos_token_id=tokenizer.eos_token_id,
            )

        for j, out in enumerate(outputs):

            input_len = inputs["attention_mask"][j].sum().item()

            gen_tokens = out[input_len:]

            decoded = tokenizer.decode(
                gen_tokens,
                skip_special_tokens=True
            )

            decoded = decoded.split("<|im_end|>")[0].strip()

            results.append(decoded)

        print(f"✅ Done: {min(i+batch_size, len(texts))}/{len(texts)}")

    return results

⏳ Load inference model...


In [15]:
print("⏳ Chuẩn bị test set...")

MAX_TEST = 300

if nest_key:

    test_sources = [
        ex[nest_key][src_col]
        for ex in test_raw
    ][:MAX_TEST]

    test_targets = [
        ex[nest_key][tgt_col]
        for ex in test_raw
    ][:MAX_TEST]

else:

    test_sources = test_raw[src_col][:MAX_TEST]
    test_targets = test_raw[tgt_col][:MAX_TEST]

print(f"✅ Test size: {len(test_sources)}")

print("⏳ Generating translations...")

translations = translate_batch(
    test_sources,
    batch_size=8
)

print(f"✅ Done: {len(translations)} translations")

import evaluate

sacrebleu_metric = evaluate.load("sacrebleu")
meteor_metric    = evaluate.load("meteor")
ter_metric       = evaluate.load("ter")

print("⏳ BLEU...")

bleu_score = round(
    sacrebleu_metric.compute(
        predictions=translations,
        references=[[t] for t in test_targets]
    )["score"],
    4
)

print("⏳ METEOR...")

meteor_score = round(
    meteor_metric.compute(
        predictions=translations,
        references=test_targets
    )["meteor"] * 100,
    4
)

print("⏳ TER...")

try:
    ter_score = round(
        ter_metric.compute(
            predictions=translations,
            references=[[t] for t in test_targets]
        )["score"],
        4
    )
except Exception as e:
    print(f"⚠️ TER Error: {e}")
    ter_score = None

print("⏳ BERTScore...")

try:
    from bert_score import score as bert_score_fn

    P, R, F1 = bert_score_fn(
        translations,
        test_targets,
        lang="vi",
        device=device,
        batch_size=16,
        verbose=False,
    )

    bert_p  = round(P.mean().item() * 100, 4)
    bert_r  = round(R.mean().item() * 100, 4)
    bert_f1 = round(F1.mean().item() * 100, 4)

except Exception as e:
    print(f"⚠️ BERTScore Error: {e}")
    bert_p = bert_r = bert_f1 = None

print("⏳ COMET...")

try:
    from comet import download_model, load_from_checkpoint

    comet_path = download_model("Unbabel/wmt22-comet-da")
    comet_model = load_from_checkpoint(comet_path)

    comet_data = [
        {
            "src": s,
            "mt": t,
            "ref": r
        }
        for s, t, r in zip(test_sources, translations, test_targets)
    ]

    comet_output = comet_model.predict(
        comet_data,
        batch_size=8,
        gpus=0   
    )

    comet_score = round(
        comet_output.system_score * 100,
        4
    )

except Exception as e:
    print(f"⚠️ COMET Error: {e}")
    comet_score = None

print("\n==============================")
print("📊 FINAL METRICS")
print("==============================")

print(f"BLEU       : {bleu_score}")
print(f"METEOR     : {meteor_score}")
print(f"TER        : {ter_score}")
print(f"BERT-P     : {bert_p}")
print(f"BERT-R     : {bert_r}")
print(f"BERT-F1    : {bert_f1}")
print(f"COMET      : {comet_score}")

print("==============================")
print("✅ DONE")

⏳ Chuẩn bị test set...
✅ Test size: 300
⏳ Generating translations...
✅ Done: 8/300
✅ Done: 16/300
✅ Done: 24/300
✅ Done: 32/300
✅ Done: 40/300
✅ Done: 48/300
✅ Done: 56/300
✅ Done: 64/300
✅ Done: 72/300
✅ Done: 80/300
✅ Done: 88/300
✅ Done: 96/300
✅ Done: 104/300
✅ Done: 112/300
✅ Done: 120/300
✅ Done: 128/300
✅ Done: 136/300
✅ Done: 144/300
✅ Done: 152/300
✅ Done: 160/300
✅ Done: 168/300
✅ Done: 176/300
✅ Done: 184/300
✅ Done: 192/300
✅ Done: 200/300
✅ Done: 208/300
✅ Done: 216/300
✅ Done: 224/300
✅ Done: 232/300
✅ Done: 240/300
✅ Done: 248/300
✅ Done: 256/300
✅ Done: 264/300
✅ Done: 272/300
✅ Done: 280/300
✅ Done: 288/300
✅ Done: 296/300
✅ Done: 300/300
✅ Done: 300 translations


[nltk_data] Downloading package wordnet to /usr/share/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /usr/share/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /usr/share/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
That's 100 lines that end in a tokenized period ('.')
It looks like you forgot to detokenize your test data, which may hurt your score.
If you insist your data is detokenized, or don't care, you can suppress this message with the `force` parameter.


⏳ BLEU...
⏳ METEOR...
⏳ TER...
⏳ BERTScore...


A parameter name that contains `beta` will be renamed internally to `bias`. Please use a different name to suppress this warning.
A parameter name that contains `gamma` will be renamed internally to `weight`. Please use a different name to suppress this warning.
A parameter name that contains `beta` will be renamed internally to `bias`. Please use a different name to suppress this warning.
A parameter name that contains `gamma` will be renamed internally to `weight`. Please use a different name to suppress this warning.
A parameter name that contains `beta` will be renamed internally to `bias`. Please use a different name to suppress this warning.
A parameter name that contains `gamma` will be renamed internally to `weight`. Please use a different name to suppress this warning.
A parameter name that contains `beta` will be renamed internally to `bias`. Please use a different name to suppress this warning.
A parameter name that contains `gamma` will be renamed internally to `weight`. Pl

⏳ COMET...


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Lightning automatically upgraded your loaded checkpoint from v1.8.3.post1 to v2.6.1. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../../root/.cache/huggingface/hub/models--Unbabel--wmt22-comet-da/snapshots/2760a223ac957f30acfb18c8aa649b01cf1d75f2/checkpoints/model.ckpt`


⚠️ COMET Error: value cannot be converted to type c10::Half without overflow

📊 FINAL METRICS
BLEU       : 19.0792
METEOR     : 54.3548
TER        : 101.284
BERT-P     : 74.6009
BERT-R     : 80.6087
BERT-F1    : 77.387
COMET      : None
✅ DONE


In [16]:
results = {
    "Metric": [
        "BLEU ↑",
        "METEOR ↑",
        "TER ↓",
        "BERTScore-P ↑",
        "BERTScore-R ↑",
        "BERTScore-F1 ↑",
        "COMET ↑",
    ],

    "Score": [

        f"{bleu_score:.2f}",

        f"{meteor_score:.2f}",

        f"{ter_score:.2f}"
        if ter_score is not None
        else "N/A",

        f"{bert_p:.2f}"
        if bert_p is not None
        else "N/A",

        f"{bert_r:.2f}"
        if bert_r is not None
        else "N/A",

        f"{bert_f1:.2f}"
        if bert_f1 is not None
        else "N/A",

        f"{comet_score:.2f}"
        if comet_score is not None
        else "N/A",
    ],

    "Description": [

        "Higher is better",

        "Higher is better",

        "Lower is better",

        "Semantic Precision",

        "Semantic Recall",

        "Semantic Similarity",

        "Neural MT Evaluation",
    ]
}

df = pd.DataFrame(results)

print("\n" + "="*75)

print("📊 LLM MACHINE TRANSLATION RESULTS (EN → VI)")

print("="*75)

print(df.to_string(index=False))

print("="*75)

print(f"\n🧠 Model     : {MODEL_NAME}")
print(f"🔧 Fine-tune : QLoRA (r=8)")
print(f"📦 Dataset   : thainq107/iwslt2015-en-vi")
print(f"📚 Train Size: {NUM_TRAIN}")
print(f"🖥️ Device    : {device}")

# Save CSV
csv_path = f"{OUTPUT_DIR}/evaluation_results.csv"

df.to_csv(csv_path, index=False)

print(f"\n✅ CSV saved:")
print(csv_path)


📊 LLM MACHINE TRANSLATION RESULTS (EN → VI)
        Metric  Score          Description
        BLEU ↑  19.08     Higher is better
      METEOR ↑  54.35     Higher is better
         TER ↓ 101.28      Lower is better
 BERTScore-P ↑  74.60   Semantic Precision
 BERTScore-R ↑  80.61      Semantic Recall
BERTScore-F1 ↑  77.39  Semantic Similarity
       COMET ↑    N/A Neural MT Evaluation

🧠 Model     : Qwen/Qwen2.5-1.5B-Instruct
🔧 Fine-tune : QLoRA (r=8)
📦 Dataset   : thainq107/iwslt2015-en-vi
📚 Train Size: 10000
🖥️ Device    : cuda

✅ CSV saved:
/kaggle/working/llm_mt_en_vi/evaluation_results.csv


In [17]:
def translate(text):
    prompt = (
        "<|im_start|>system\n"
        "You are a professional translator. Translate English to Vietnamese.<|im_end|>\n"
        "<|im_start|>user\n"
        f"Translate to Vietnamese:\n{text}<|im_end|>\n"
        "<|im_start|>assistant\n"
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=256
    ).to(device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=128,
            num_beams=4,
            do_sample=False
        )

    result = tokenizer.decode(output[0], skip_special_tokens=True)

    result = result.split("assistant")[-1].strip()

    return result

print(translate("Artificial intelligence is changing the world."))
print(translate("I love learning deep learning models."))
print(translate("This is a very powerful language model."))

Công nghệ trí tuệ nhân tạo đang thay đổi thế giới .
Tôi thích học các mô hình học sâu .
Đây là một mô hình ngôn ngữ rất mạnh mẽ .


In [18]:
import shutil

MODEL_PATH = f"{OUTPUT_DIR}/merged_model"
ZIP_PATH = "/kaggle/working/llm_mt_model.zip"

shutil.make_archive(
    base_name="/kaggle/working/llm_mt_model",
    format="zip",
    root_dir=MODEL_PATH
)

print("✅ Model đã được zip xong!")
print(ZIP_PATH)

✅ Model đã được zip xong!
/kaggle/working/llm_mt_model.zip
